# Manuscript figures

Code accompanying *Boisvert-Vigneault et al. (2026), "Simulating locked pastures in snow from Arctic rain events"*.
Run `Pre-process_datasets.ipynb` first: it writes the files read here.

| Figure | Content | Output file |
|---|---|---|
| Fig. 2 | Water holding capacity as a function of snow density | `fig02_water_holding_capacity.png` |
| Fig. 3 | Wind speed Q-Q plots, reanalyses vs. weather stations | `fig03_wind_qqplot.png` |
| Fig. 4 | Cumulative precipitation at Sachs Harbour (a: annual, b: monthly) | `fig04_cumulative_precip_sachs.png` |
| Fig. 5 | FBI and ETS of the precipitation phase partitioning methods | `fig05_ppm_fbi_ets.png` |
| Fig. 6 | Solid vs. liquid precipitation during the October 2003 ROS event, Thomsen River | `fig06_precip_phase_ros2003.png` |
| Fig. 8 | Basal 20 mm density by physical scheme (normal, cold, no-rain ensembles) | `fig08_basal20mm_density_boxplots.png` |
| Fig. D1 (Appendix D) | As Fig. 6b for every reanalysis-PPM pair | `figD1_precip_phase_all_pairs.png` |
| Supplement | As Fig. 6b for other ROS events (1988-2018) | `supplement/ros_event_<year>.png` |

Not produced here: Fig. 1 (map, QGIS) and Figs. 7, 9, 10 and 11 (SVS2-Crocus output).

**Requirements:** numpy, pandas (>= 2.0), xarray, netCDF4, scipy, matplotlib, seaborn, metpy.
Folders are relative to `PROJECT_DIR` (environment variable `ROS_PROJECT_DIR`, default: current directory), with the same layout as in `Pre-process_datasets.ipynb`.

# Setup

In [ ]:
import calendar
import os
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from scipy import stats

## Paths and parameters

In [ ]:
PROJECT_DIR = Path(os.environ.get('ROS_PROJECT_DIR', Path.cwd()))

LWS_DIR = PROJECT_DIR / 'Data' / 'RAW' / 'Banks_weather_stations'
RAW_MET_DIR = PROJECT_DIR / 'Data' / 'RAW' / 'met_extractions'
PPM_NETCDF = PROJECT_DIR / 'Data' / 'PROCESSED' / 'PPM' / 'basin_forcing_PPMs_1980-2020_elevcorrF.nc'
RESULTS_DIR = PROJECT_DIR / 'Results'
FIG_DIR = PROJECT_DIR / 'final_figures'
(FIG_DIR / 'supplement').mkdir(parents=True, exist_ok=True)

# SVS2-Crocus ensemble summaries used in Fig. 8 (one row per ensemble member)
ENSEMBLE_SUMMARY_FILES = {'normal': RESULTS_DIR / 'TR_lockedpasture_summary_2003-10-31.csv',
                          'cold': RESULTS_DIR / 'cold_lockedpasture_summary_2003-10-31.csv',
                          'norain': RESULTS_DIR / 'norain_lockedpasture_summary_2003-10-31.csv'}

# Names, labels and colours shared by several figures
REANALYSES = ['ERA5', 'CaSR32_analysis', 'CaSR32_prediction']
REANALYSIS_LABELS = {'ERA5': 'ERA5', 'CaSR32_analysis': 'CaSRv3.2 - Reanalysis', 'CaSR32_prediction': 'CaSRv3.2 - Forecast'}
REANALYSIS_COLORS = {'ERA5': '#b02e0c', 'CaSR32_analysis': '#759ca4', 'CaSR32_prediction': '#c89434'}
PHASE_COLORS = {'Rain': '#52aa8a', 'Snow': '#bb9e7d', 'Ice Pellets': '#786452', 'Freezing Rain': '#172a3a'}
TEMPERATURE_COLOR = '#dd614a'

# Remote-sensing ROS detections at Thomsen River, October 2003 (Langlois et al., 2026)
ROS_DETECTIONS_2003 = [(pd.Timestamp('2003-10-02 12:00'), pd.Timestamp('2003-10-03 00:00')),
                       (pd.Timestamp('2003-10-04 00:00'), pd.Timestamp('2003-10-05 12:00'))]
ROS_WINDOW_2003 = (pd.Timestamp('2003-09-26'), pd.Timestamp('2003-10-10'))   # x-axis range of Figs. 6 and D1

## Plot style and helper functions

In [ ]:
MANUSCRIPT_RC = {
    'font.family': 'sans-serif',
    'font.sans-serif': ['Times New Roman', 'Helvetica', 'DejaVu Sans'],   # fallback order
    'font.size': 10, 'font.style': 'normal', 'font.weight': 'normal',
    'axes.titlesize': 14, 'axes.titleweight': 'bold', 'axes.labelsize': 10, 'axes.labelweight': 'normal',
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'legend.title_fontsize': 11,
    'figure.titlesize': 14, 'figure.titleweight': 'bold',
}


def use_manuscript_style(keep_ggplot_sizes=True):
    """ggplot style with the manuscript fonts. Figs. 2-6 and D1 keep the font and axis-label sizes of the ggplot
    style; Fig. 8 overrides them with MANUSCRIPT_RC (keep_ggplot_sizes=False)."""
    plt.style.use('ggplot')
    rc = dict(MANUSCRIPT_RC)
    if keep_ggplot_sizes:
        rc = {key: value for key, value in rc.items() if key not in mpl.style.library['ggplot']}
    mpl.rcParams.update(rc)


def save_figure(fig, filename, **kwargs):
    path = FIG_DIR / filename
    fig.savefig(path, **kwargs)
    print(f'Saved {path}')


def shade_ros_detections(ax, detections, **kwargs):
    """Shade the ROS detection periods; only the first span is labelled in the legend."""
    label = kwargs.pop('label', None)
    for i, (start, end) in enumerate(detections):
        ax.axvspan(mdates.date2num(start), mdates.date2num(end), label=label if i == 0 else None, **kwargs)

In [ ]:
# ----- Weather-station and reanalysis data ---------------------------------------------

def load_lws_stations(folder):
    """Hourly ECCC station data (*_8020.csv), as {station: DataFrame(time, weather, pressure, rel_hum, temp, wind_spd)}."""
    stations = {}
    for csv in sorted(Path(folder).glob('*_8020.csv')):
        df = pd.read_csv(csv, low_memory=False)
        station = df['station_name'][0].replace(' ', '_')
        if 'SACHS' in station:
            station = 'SACHS_A'
        df = df[['time', 'weather', 'pressure', 'rel_hum', 'temp', 'wind_spd']].copy()
        df['time'] = pd.to_datetime(df['time'], format='mixed')
        stations[station] = df
    return stations


def load_adjusted_daily_precip(path):
    """Adjusted Daily Rainfall and Snowfall Data for Canada (Wang et al., 2017), Sachs Harbour.
    Returns daily total precipitation (mm) indexed by date; missing days (negative flags) are dropped."""
    df = pd.read_csv(path, sep=';')
    total = df['cR+cS(mm)'].astype(str).str.lower().str.replace('[a-z]', '', regex=True).astype(float)   # strip quality flags
    dates = pd.date_range('1955-11-01', '2013-10-31', freq='D')
    assert len(dates) == len(df), 'The adjusted daily file no longer matches the 1955-11-01 to 2013-10-31 date range'
    series = pd.Series(total.to_numpy(), index=dates, name='Total_precip')
    return series[series >= 0]


def sync_series(obs_values, obs_time, model_values, model_time):
    """Keep the timestamps present in both series, within their common time window.
    Returns a DataFrame with columns time, obs, model."""
    obs = pd.DataFrame({'time': pd.to_datetime(obs_time), 'obs': np.asarray(obs_values)})
    model = pd.DataFrame({'time': pd.to_datetime(model_time), 'model': np.asarray(model_values)})
    start = max(obs['time'].min(), model['time'].min())
    end = min(obs['time'].max(), model['time'].max())
    obs = obs[(obs['time'] >= start) & (obs['time'] <= end)]
    model = model[(model['time'] >= start) & (model['time'] <= end)]
    return obs.merge(model, on='time', how='inner').sort_values('time').reset_index(drop=True)


# ----- Raw forcing files (CaSRv3.2 extraction with the B00 four-phase precipitation) ----

def met_timestamps(df):
    """Build timestamps from the YEAR, JDAY, HOUR and MINS columns of a .met table."""
    return (pd.to_datetime(df['YEAR'], format='%Y')
            + pd.to_timedelta(df['JDAY'] - 1, unit='D')
            + pd.to_timedelta(df['HOUR'], unit='h')
            + pd.to_timedelta(df['MINS'], unit='m'))


def read_b00_event_data(product, station='THOMSEN_RIVER'):
    """Hourly B00 precipitation by phase (mm/h) and air temperature (deg C) from the raw CaSRv3.2 extraction.
    product: 'analysis' (CaSRv3.2-R) or 'prediction' (CaSRv3.2-F)."""
    folder = RAW_MET_DIR / f'CaSR32_{product}'
    phases = pd.read_csv(folder / f'extra_pcpn_casr3_{station}_8010_casr32_forcing.met', header=0,
                         names=['HOUR', 'MINS', 'JDAY', 'YEAR', 'Rain', 'Snow', 'Ice Pellets', 'Freezing Rain'])
    forcing = pd.read_csv(folder / f'basin_forcing_{station}_8010_casr32_forcing.met', sep=r'\s+', header=None,
                          usecols=range(11), names=['HOUR', 'MINS', 'JDAY', 'YEAR', 'FSIN', 'FLIN', 'PRE', 'TA', 'QA', 'UV', 'PRES'])
    df = phases[['Rain', 'Snow', 'Ice Pellets', 'Freezing Rain']] * 3600     # kg m-2 s-1 -> mm per hour
    df['Temperature'] = forcing['TA'].to_numpy()
    return df.set_index(met_timestamps(phases))

## Load the data

Used by Figs. 3-6, D1 and the supplement (Figs. 2 and 8 do not need it).

In [ ]:
ppm_ds = xr.open_dataset(PPM_NETCDF)
lws = load_lws_stations(LWS_DIR)
adj_daily = load_adjusted_daily_precip(LWS_DIR / '2503650_AdjDly.csv')
casr_r = read_b00_event_data('analysis')       # CaSRv3.2-R with B00 at Thomsen River: four phases and air temperature


def lws_temperature(station, start, end):
    """Hourly weather-station air temperature between `start` and `end`."""
    df = lws[station][['time', 'temp']]
    return df[(df['time'] >= start) & (df['time'] <= end)]

# Figure 2. Water holding capacity as a function of snow density

In [ ]:
RHO_WATER, RHO_ICE = 999, 918    # kg m-3


def max_water_b92(rho_snow, rho_water=RHO_WATER, rho_ice=RHO_ICE):
    """Brun et al. (1992): 5% of the pore volume."""
    return 0.05 * rho_water * (1 - np.asarray(rho_snow) / rho_ice)


def max_water_o04(rho_snow, rho_water=RHO_WATER, rho_ice=RHO_ICE):
    """Oleson et al. (2004): 3.3% of the pore volume."""
    return 0.033 * rho_water * (1 - np.asarray(rho_snow) / rho_ice)


def max_water_b02(rho_snow, r_min=0.03, r_max=0.1, rho_ref=200):
    """Boone (2002): maximal liquid water mass fraction, decreasing from r_max to r_min up to rho_ref."""
    rho = np.asarray(rho_snow, dtype=float)
    return rho * (r_min + (r_max - r_min) * np.maximum(0, (rho_ref - rho) / rho_ref))


def max_water_spk(rho_snow, rho_water=RHO_WATER, rho_ice=RHO_ICE):
    """Wever et al. (2014), as implemented in Crocus: a function of the porosity."""
    rho = np.asarray(rho_snow, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        w_low_density = (rho_ice * rho_water * (0.083069 - 0.1023 * rho / rho_ice)
                         / (rho_ice - 0.1023 * rho_water))
        phi = 1 - (rho - w_low_density) / rho_ice
        w_high_density = rho_water * (0.0264 + 0.0099 * phi / (1 - phi))
    return np.select([phi >= 0.77, phi < 0.188], [w_low_density, 0.0], default=w_high_density)

In [ ]:
use_manuscript_style()
colors = ['#b02e0c', '#d78f09', '#6a868a', '#220901']
rho_snow = np.linspace(50, RHO_ICE, 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rho_snow, max_water_b02(rho_snow), label='B02', color=colors[0])
ax.plot(rho_snow, max_water_b92(rho_snow), label='B92', color=colors[1])
ax.plot(rho_snow, max_water_spk(rho_snow), label='SPK', color=colors[2])
ax.plot(rho_snow, max_water_o04(rho_snow), label='O04', color=colors[3])
ax.axvline(RHO_ICE, c='k', alpha=0.5, linestyle='dotted')       # density of ice
ax.text(x = 918, y = 40, s = 'Ice density = 918 kg m${^-3}$', rotation = -90)
ax.set_xlim(0, 1000)
ax.xaxis.set_major_locator(MultipleLocator(100))
ax.set_xlabel('Snow density (kg m${^-3}$)')
ax.set_ylabel('Volumetric liquid water holding capacity (kg m${^-3}$)')
ax.legend(ncol=4, loc='upper center')

save_figure(fig, 'fig02_water_holding_capacity.png', bbox_inches='tight')
plt.show()

# Figure 3. Wind speed Q-Q plots

In [ ]:
def hourly_wind_pairs(ppm_ds, lws, reanalyses=('CaSR32_analysis', 'ERA5')):
    """Hourly pairs (obs, model) of 10-m wind speed in m/s for each reanalysis and station.
    The station reports km/h. Only wind speed is needed here, which does not depend on the PPM."""
    base = ppm_ds.sel(PPM='HP13')
    pairs = {}
    for reanalysis in reanalyses:
        station_pairs = []
        for station in base.location.values:
            ds = base.sel(forecast=reanalysis, location=station)
            df = sync_series(lws[station]['wind_spd'].to_numpy(), lws[station]['time'].to_numpy(),
                             ds['UV'].values, ds.time.values)
            df['obs'] = df['obs'] / 3.6
            df = df.dropna()
            df['station'] = station
            station_pairs.append(df)
        pairs[reanalysis] = pd.concat(station_pairs, ignore_index=True)
    return pairs


wind_pairs = hourly_wind_pairs(ppm_ds, lws)

In [ ]:
use_manuscript_style()
reanalysis_titles = {'CaSR32_analysis': 'CaSRv3.2', 'ERA5': 'ERA5'}
reanalysis_colors = {'CaSR32_analysis': 'teal', 'ERA5': 'firebrick'}
station_titles = {'SACHS_A': 'Sachs Harbour A', 'THOMSEN_RIVER': 'Thomsen River', 'NANGMAGVIK_LAKE': 'Nangmagvik Lake'}
quantile_levels = np.linspace(0.01, 0.99, 99)      # 1st to 99th percentile

fig, axes = plt.subplots(nrows=len(reanalysis_titles), ncols=len(station_titles), sharex=True, sharey=True,
                         figsize=(8, 5))

for i, reanalysis in enumerate(reanalysis_titles):
    for j, station in enumerate(station_titles):
        pairs = wind_pairs[reanalysis][wind_pairs[reanalysis]['station'] == station]
        obs_quantiles = np.quantile(pairs['obs'], quantile_levels)
        model_quantiles = np.quantile(pairs['model'], quantile_levels)
        ax = axes[i, j]

        ax.scatter(obs_quantiles, model_quantiles, color=reanalysis_colors[reanalysis], edgecolor='darkslategrey',
                   alpha=0.7, zorder=2)
        ax.axline((0, 0), slope=1, color='k', linestyle='--', label='y = x', zorder=1)    # perfect agreement
        ax.xaxis.set_major_locator(MultipleLocator(4))
        ax.yaxis.set_major_locator(MultipleLocator(4))
        lower_limit = min(obs_quantiles.min(), model_quantiles.min()) - 0.5
        ax.set_xlim(lower_limit, 14.5)
        ax.set_ylim(lower_limit, 14.5)

        if i == 0:
            ax.set_title(station_titles[station], fontsize=12, fontweight='semibold')
        if j == 0:
            ax.text(-0.23, 0.5, reanalysis_titles[reanalysis], ha='right', va='center', fontsize=12,
                    fontweight='semibold', rotation=90, transform=ax.transAxes)

axes[1, 1].set_xlabel('Measured Wind Speed Quantiles (ms$^{-1}$)', fontsize=11)
axes[1, 0].text(-0.22, 1, 'Modelled Wind Speed Quantiles (ms$^{-1}$)', va='center', fontsize=11, color='dimgrey',
                rotation=90, transform=axes[1, 0].transAxes)
fig.tight_layout()

save_figure(fig, 'fig03_wind_qqplot.png', bbox_inches='tight')
plt.show()

# Figure 4. Cumulative precipitation at Sachs Harbour

(a) Total annual precipitation (complete years only); (b) mean monthly precipitation (complete months only).
Reanalysis precipitation is aggregated over exactly the same days as the adjusted station record.

In [ ]:
TRACE_PRECIP_MM = 0.2      # mm/day: days below this amount are set to zero in the reanalyses


def daily_reanalysis_precip(ppm_ds, station='SACHS_A'):
    """Daily total precipitation (mm) per reanalysis at `station`."""
    daily = {}
    for reanalysis in ppm_ds.forecast.values:
        pre = ppm_ds['PRE'].sel(forecast=reanalysis, location=station, PPM='HP13')
        hourly_mm = pd.Series(pre.values * 3600, index=pd.to_datetime(pre.time.values))   # kg m-2 s-1 -> mm per hour
        series = hourly_mm.resample('1D').sum()
        series[series < TRACE_PRECIP_MM] = 0
        daily[reanalysis] = series
    return daily


def complete_periods_only(daily, freq):
    """Keep only the days of fully reported months (freq='month') or years (freq='year')."""
    if freq == 'month':
        n_reported = daily.groupby(pd.Grouper(freq='MS')).transform('count')
        return daily[n_reported.to_numpy() == daily.index.days_in_month]
    complete_years = [year for year, days in daily.groupby(daily.index.year)
                      if len(days) == (366 if calendar.isleap(year) else 365)]
    return daily[daily.index.year.isin(complete_years)]


def aggregate_precip(obs_daily, model_daily, freq):
    """Annual totals (freq='year') or mean monthly totals by calendar month (freq='month') of the station record
    ('LWS') and of each reanalysis, over the periods that are complete in the station record.

    Monthly means average only the complete months ('n_periods' gives their number for each calendar month);
    months missing from the station record are not counted as zero precipitation."""
    start = max(obs_daily.index[0], *(s.index[0] for s in model_daily.values()))
    end = min(obs_daily.index[-1], *(s.index[-1] for s in model_daily.values()))
    obs = complete_periods_only(obs_daily.loc[start:end], freq)
    period = 'M' if freq == 'month' else 'Y'
    valid_periods = obs.index.to_period(period).unique()

    totals = {}
    for name, daily in {'LWS': obs, **{r: s.loc[start:end] for r, s in model_daily.items()}}.items():
        daily = daily[daily.index.to_period(period).isin(valid_periods)]
        totals[name] = daily.groupby(daily.index.to_period(period)).sum()       # one total per valid period
    totals = pd.DataFrame(totals)

    if freq == 'year':
        totals.index = totals.index.year
        return totals.reindex(range(start.year, end.year + 1))
    by_month = totals.groupby(totals.index.month)
    table = by_month.mean()
    table['n_periods'] = by_month['LWS'].count()
    return table.reindex(range(1, 13))

In [ ]:
model_daily = daily_reanalysis_precip(ppm_ds)

monthly = aggregate_precip(adj_daily, model_daily, 'month')
annual = aggregate_precip(adj_daily, model_daily, 'year').dropna()

print(f'Complete years used in panel (a): {list(annual.index)}')
print('\nMean monthly precipitation (mm) and number of complete months averaged (panel b)')
print(monthly.round(1).to_string())
print(f'\nSum of the monthly means, LWS: {monthly["LWS"].sum():.0f} mm '
      f'(mean annual total of the complete years: {annual["LWS"].mean():.0f} mm)')
print('\nTotal annual precipitation (mm)')
print(annual.round(1).to_string())

In [ ]:
use_manuscript_style()
product_order = ['LWS', 'CaSR32_prediction', 'CaSR32_analysis', 'ERA5']
labels = {'LWS': 'LWS', 'CaSR32_prediction': 'CaSRv3.2 - Forecast', 'CaSR32_analysis': 'CaSRv3.2 - Reanalysis', 'ERA5': 'ERA5'}
box_colors = {labels[k]: v for k, v in {**REANALYSIS_COLORS, 'LWS': 'darkgrey'}.items()}
line_colors = {labels[k]: v for k, v in {**REANALYSIS_COLORS, 'LWS': 'k'}.items()}

fig, (ax_annual, ax_monthly) = plt.subplots(2, 1, figsize=(9, 9))

# (a) annual totals
annual_totals = annual[product_order].rename(columns=labels)
sns.boxplot(data=annual_totals, ax=ax_annual, palette=box_colors, order=list(labels.values()))
ax_annual.set_ylabel('Annual cumulative precipitation (mm)')
ax_annual.tick_params(axis='x', labelsize=10)
ax_annual.text(-0.1, 1.06, '(a)', ha='left', va='top', fontsize=14, transform=ax_annual.transAxes)

# (b) mean monthly totals
for product in product_order:
    is_lws = product == 'LWS'
    ax_monthly.plot(monthly.index, monthly[product], label=labels[product], color=line_colors[labels[product]],
                    linewidth=1.7 if is_lws else 2.3, linestyle='--' if is_lws else '-')
ax_monthly.set_ylim(0, 33)
handles, legend_labels = ax_monthly.get_legend_handles_labels()
legend = ax_monthly.legend(handles, legend_labels, bbox_to_anchor=(0.5, 0.99), loc='upper center', ncol=4)
legend.get_frame().set_facecolor('white')
ax_monthly.set_ylabel('Monthly cumulative precipitation (mm)')
ax_monthly.set_xticks(range(1, 13))
ax_monthly.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax_monthly.text(-0.1, 1.06, '(b)', ha='left', va='top', fontsize=14, transform=ax_monthly.transAxes)

fig.tight_layout()
save_figure(fig, 'fig04_cumulative_precip_sachs.png')
plt.show()

## Daily precipitation statistics (all temperatures)

In [ ]:
rows = []
for reanalysis, series in model_daily.items():
    pair = sync_series(adj_daily.to_numpy(), adj_daily.index, series.to_numpy(), series.index)
    regression = stats.linregress(pair['obs'], pair['model'])
    rows.append({'reanalysis': reanalysis, 'n_days': len(pair),
                 'rmse_mm': np.sqrt(np.mean((pair['model'] - pair['obs']) ** 2)),
                 'r2': regression.rvalue ** 2, 'slope': regression.slope})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# Figure 5. Precipitation phase detection skill (FBI and ETS)

Near-freezing periods at Sachs Harbour. The scores are computed in `Pre-process_datasets.ipynb` (Section 5) and read from `ppm_phase_scores.csv`.
The ETS is identical for solid and liquid precipitation, so only the solid bar is drawn.

In [ ]:
PPMS_FIG5 = ['J18', 'HP13', 'W19', 'B00']
REANALYSIS_TICKS = {'ERA5': 'ERA5', 'CaSR32_analysis': 'CaSRv3.2-R', 'CaSR32_prediction': 'CaSRv3.2-F'}

scores = pd.read_csv(RESULTS_DIR / 'ppm_phase_scores.csv').set_index(['ppm', 'reanalysis', 'phase'])


def score(metric, ppm, reanalysis, phase):
    """Score of one reanalysis-PPM pair; NaN where the pair does not exist (ERA5 with B00)."""
    try:
        return scores.loc[(ppm, reanalysis, phase), metric]
    except KeyError:
        return np.nan


use_manuscript_style()
bar_width = 0.27
x = np.arange(len(REANALYSES))

fig, axes = plt.subplots(2, len(PPMS_FIG5), figsize=(2.7 * len(PPMS_FIG5), 5), sharex=True, sharey='row')

for col, ppm in enumerate(PPMS_FIG5):
    ax_fbi, ax_ets = axes[0, col], axes[1, col]
    for i, reanalysis in enumerate(REANALYSES):
        color = REANALYSIS_COLORS[reanalysis]
        ax_fbi.bar(x[i] - bar_width / 2, score('FBI', ppm, reanalysis, 'Solid'), width=bar_width, color=color,
                   edgecolor='black', alpha=0.85)
        ax_fbi.bar(x[i] + bar_width / 2, score('FBI', ppm, reanalysis, 'Liquid'), width=bar_width, color=color,
                   edgecolor='black', hatch='///', alpha=0.85)
        ax_ets.bar(x[i], score('ETS', ppm, reanalysis, 'Solid'), width=bar_width, color=color,
                   edgecolor='black', alpha=0.85)

    for ax in (ax_fbi, ax_ets):
        ax.axhline(1, color='black', linestyle='--', linewidth=1)      # perfect score
    ax_fbi.yaxis.set_major_locator(MultipleLocator(0.4))
    ax_fbi.set_ylim(0, 1.6)
    ax_fbi.set_title(ppm)
    ax_ets.set_xticks(x)
    ax_ets.set_xticklabels([REANALYSIS_TICKS[r] for r in REANALYSES])

axes[0, 0].set_ylabel('FBI')
axes[1, 0].set_ylabel('ETS')

legend_handles = [Line2D([0], [0], label='Perfect score', color='k', linestyle='dashed', linewidth=2),
                  Patch(facecolor='white', edgecolor='black', label='Solid Precipitation'),
                  Patch(facecolor='white', edgecolor='black', hatch='///', label='Liquid Precipitation')]
fig.legend(handles=legend_handles, loc='upper center', ncol=len(REANALYSES) + 2, bbox_to_anchor=(0.5, 0), fontsize=9)
fig.tight_layout()

save_figure(fig, 'fig05_ppm_fbi_ets.png', dpi=200, bbox_inches='tight')
plt.show()

# Figure 6. Solid vs. liquid precipitation during the October 2003 ROS event (Thomsen River)

(a) Cumulative solid (continuous) and liquid (dashed) precipitation of ERA5 with HP13 and of CaSRv3.2 with B00;
(b) CaSRv3.2-R with B00 by phase, in 12-hour accumulations. The grey/red bands are the remote-sensing ROS detections.

In [ ]:
start, end = ROS_WINDOW_2003
thomsen_ds = ppm_ds.sel(location='THOMSEN_RIVER', time=slice(start, end))
thomsen_lws = lws_temperature('THOMSEN_RIVER', start, end)
casr_r_event = casr_r.loc[start:end]

In [ ]:
use_manuscript_style()
displayed = {'ERA5': 'HP13', 'CaSR32_prediction': 'B00', 'CaSR32_analysis': 'B00'}   # reanalysis -> PPM shown in (a)

fig = plt.figure(figsize=(10, 10))

# ---- (a) cumulative precipitation -------------------------------------------------------
with plt.style.context('default'):
    ax1 = fig.add_subplot(2, 1, 1)
    for reanalysis, ppm in displayed.items():
        ds = thomsen_ds.sel(forecast=reanalysis, PPM=ppm)
        time = pd.to_datetime(ds.time.values)
        liquid = pd.Series(ds['PRERN'].values * 3600, index=time).cumsum()        # kg m-2 s-1 -> mm per hour
        solid = pd.Series(ds['PRESNO'].values * 3600, index=time).cumsum()
        color = REANALYSIS_COLORS[reanalysis]
        label = f'{REANALYSIS_LABELS[reanalysis]}: {ppm}'
        ax1.plot(liquid.index, liquid, color=color, zorder=3, linestyle='dashed', linewidth=2)
        ax1.plot(solid.index, solid, color=color, zorder=3, linestyle='solid', linewidth=2, label=label)

    ax1.set_ylim([-1, 18])
    ax1.set_ylabel('Cumulative precipitation (mm)')
    ax1.text(-0.07, 1.12, '(a)', ha='left', va='top', fontsize=14, transform=ax1.transAxes)

    ax1_twin = ax1.twinx()
    ax1_twin.set_ylim([-21, 5])
    ax1_twin.fill_between(thomsen_lws['time'], thomsen_lws['temp'], -21, color='red', zorder=4, alpha=0.1)
    ax1_twin.plot(thomsen_lws['time'], thomsen_lws['temp'], color='red', zorder=4, alpha=0.2)
    ax1_twin.set_ylabel('Temperature (°C)')
    ax1_twin.grid(False)
    ax1_twin.patch.set_visible(False)
    ax1_twin.axhline(0, zorder=1, color='k', linestyle='dotted', alpha=0.3)

    shade_ros_detections(ax1, ROS_DETECTIONS_2003, color='grey', alpha=0.16, zorder=6)

    line_handles, _ = ax1.get_legend_handles_labels()          # ERA5, CaSRv3.2-F, CaSRv3.2-R
    temperature_patch = mpatches.Patch(color='red', alpha=0.2, label='LWS Temperature')
    ros_patch = mpatches.Patch(color='grey', alpha=0.3, label='ROS detections')
    liquid_line = Line2D([0], [0], label='Liquid', color='k', linestyle='dashed', linewidth=2)
    solid_line = Line2D([0], [0], label='Solid', color='k', linestyle='solid', linewidth=2)
    ax1.legend(handles=[line_handles[0], solid_line, line_handles[1], liquid_line, line_handles[2],
                        ros_patch, temperature_patch],
               bbox_to_anchor=(0.5, 1.125), loc='upper center', ncol=4, frameon=False)

# ---- (b) precipitation by phase, CaSRv3.2-R + B00 ---------------------------------------------
with plt.style.context('ggplot'):
    ax2 = fig.add_subplot(2, 1, 2, sharex=ax1)
    phases_12h = casr_r_event.drop(columns='Temperature').groupby(pd.Grouper(freq='12h')).sum()
    bottom = np.zeros(len(phases_12h))
    for phase in phases_12h.columns:
        ax2.bar(mdates.date2num(phases_12h.index), phases_12h[phase], bottom=bottom, width=0.45, label=phase,
                color=PHASE_COLORS[phase], align='edge', zorder=3)
        bottom += phases_12h[phase].to_numpy()

    ax2.set_ylim([0, 4])
    ax2.set_ylabel('Precipitation (mm)')
    ax2.grid(True, axis='y', zorder=1, alpha=0.4)
    ax2.text(-0.07, 1.12, '(b)', ha='left', va='top', fontsize=14, transform=ax2.transAxes)

    ax2_twin = ax2.twinx()
    ax2_twin.plot(casr_r_event.index, casr_r_event['Temperature'], color=TEMPERATURE_COLOR, zorder=4,
                  label='CaSRv3.2 Temperature')
    ax2_twin.plot(thomsen_lws['time'], thomsen_lws['temp'], color=TEMPERATURE_COLOR, zorder=4, linestyle='--',
                  label='LWS Temperature')
    ax2_twin.set_ylabel('Temperature (°C)')
    ax2_twin.grid(False)
    ax2_twin.patch.set_visible(False)
    ax2_twin.axhline(0, zorder=1, color='k', linestyle='--', alpha=0.3)

    shade_ros_detections(ax2, ROS_DETECTIONS_2003, color='red', alpha=0.16, label='ROS detections')
    ax2.legend(bbox_to_anchor=(0.5, 1.065), loc='upper center', ncol=5, frameon=False)
    ax2_twin.legend(bbox_to_anchor=(0.5, 1.115), loc='upper center', ncol=2, frameon=False)

for ax in (ax1, ax2):
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    for tick_label in ax.get_xticklabels(which='major'):
        tick_label.set(rotation=0, horizontalalignment='center')

fig.tight_layout()
save_figure(fig, 'fig06_precip_phase_ros2003.png')
plt.show()

## Numbers quoted in Section 3.2

In [ ]:
event = casr_r.loc['2003-09-19':'2003-10-10']
print('CaSRv3.2-R with B00, 19 September - 10 October 2003, Thomsen River')
print(f'  rain: {event["Rain"].sum():.2f} mm, freezing rain: {event["Freezing Rain"].sum():.2f} mm, '
      f'total liquid: {(event["Rain"] + event["Freezing Rain"]).sum():.2f} mm')

after = casr_r.loc['2003-10-03':'2003-10-10']
above_zero = after[after['Temperature'] > 0]
print(f'  air temperature above 0 °C from {above_zero.index[0]:%d %B %H:%M} to {above_zero.index[-1]:%d %B %H:%M}, '
      f'maximum {after["Temperature"].max():.1f} °C')

# Figure 8. Basal 20 mm density by physical scheme

Boxes: bottom 20 mm density of the 108 ensemble members, grouped by the option chosen for each scheme, for the normal, cold and no-rain ensembles.
Labels: share of members above 350 kg m-3 (a locked pasture).

In [ ]:
DENSITY_THRESHOLD = 350        # kg m-3
PANELS = [('HOLD', 'Water holding capacity'), ('FALL', 'Falling snow density'),
          ('DRIFT', 'Wind packing'), ('COMP', 'Snow compaction')]
OPTION_ORDER = {'HOLD': ['B92', 'B02', 'O04', 'SPK'], 'FALL': ['R21', 'GW1', 'GW2'],
                'DRIFT': ['R21W', 'R21R', 'R21F'], 'COMP': ['B92', 'S14', 'T11']}
SCENARIOS = [('normal', 'Normal', '#64906e'), ('cold', 'Cold', '#c45a63'), ('norain', 'No-rain', '#502419')]
BOX_OFFSETS = {'normal': -0.25, 'cold': 0.0, 'norain': 0.25}
BOX_WIDTH = 0.20
DENSITY_COLUMN_HINTS = ['basal_2cm_density_kg_m3', 'basal_2cm_density', 'density_2cm',
                        'basal_20mm_density', 'bottom_20mm_density']


def read_ensemble_table(path):
    """Option of every scheme and bottom 20 mm density (kg m-3) of each ensemble member."""
    df = pd.read_csv(path, sep=None, engine='python', encoding='utf-8-sig')
    df.columns = [str(c).strip() for c in df.columns]
    density_column = next((c for c in DENSITY_COLUMN_HINTS if c in df.columns), None)
    if density_column is None:
        raise KeyError(f'No bottom 20 mm density column in {path.name}; columns: {list(df.columns)}')
    table = df[[column for column, _ in PANELS]].astype(str).apply(lambda s: s.str.strip())
    density = df[density_column]
    if not pd.api.types.is_numeric_dtype(density):
        density = pd.to_numeric(density.astype(str).str.replace(',', '.', regex=False), errors='coerce')   # decimal commas
    table['density'] = density.astype(float)
    print(f'{path.name}: {len(table)} members, density column "{density_column}"')
    return table


def percent_above(values, threshold=DENSITY_THRESHOLD):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return 100.0 * np.sum(values > threshold) / len(values) if len(values) else np.nan


ensemble_tables = {name: read_ensemble_table(path) for name, path in ENSEMBLE_SUMMARY_FILES.items()}

In [ ]:
use_manuscript_style(keep_ggplot_sizes=False)
mpl.rcParams.update({'axes.titleweight': 'normal', 'font.sans-serif': ['Times New Roman', 'Helvetica', 'Arial', 'DejaVu Sans']})

fig, axes = plt.subplots(len(PANELS), 1, sharey=True, figsize=(8, 10))
left, right, bottom_margin, top_margin = 0.11, 0.97, 0.06, 0.94
fig.subplots_adjust(left=left, right=right, bottom=bottom_margin, top=top_margin, hspace=0.32)

for ax, (scheme, title), letter in zip(axes, PANELS, 'abcd'):
    options = OPTION_ORDER[scheme]
    ax.set_axisbelow(True)

    for key, _, color in SCENARIOS:
        table = ensemble_tables[key]
        data, positions = [], []
        for i, option in enumerate(options):
            values = table.loc[table[scheme] == option, 'density'].dropna().to_numpy()
            if len(values) == 0:
                continue
            data.append(values)
            positions.append(i + BOX_OFFSETS[key])
            ax.annotate(f'{percent_above(values):.0f}%', xy=(i + BOX_OFFSETS[key], values.max()), xytext=(0, 2),
                        textcoords='offset points', ha='center', va='bottom', fontsize=9, fontweight='bold',
                        color='black', zorder=5,
                        bbox=dict(facecolor=ax.get_facecolor(), edgecolor='none', pad=0.6))   # keeps the label readable over the threshold line
        if data:
            ax.boxplot(data, positions=positions, widths=BOX_WIDTH, patch_artist=True, manage_ticks=False,
                       boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.8),
                       medianprops=dict(color='black', linewidth=1.2),
                       whiskerprops=dict(color='black', linewidth=0.8), capprops=dict(color='black', linewidth=0.8),
                       flierprops=dict(marker='o', markersize=2.5, markerfacecolor=color, markeredgecolor='black',
                                       markeredgewidth=0.4, alpha=0.8),
                       zorder=3)

    ax.axhline(DENSITY_THRESHOLD, color='black', linestyle='--', linewidth=1.2, zorder=1)
    ax.set_xticks(range(len(options)))
    ax.set_xticklabels(options)
    ax.set_xlim(-0.6, len(options) - 0.4)
    ax.tick_params(axis='x', length=4, color='black')
    ax.set_ylabel('Bottom 20 mm density\n(kg m$^{-3}$)')
    ax.grid(False, axis='x')
    ax.set_title(f'({letter}) {title}', loc='left')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')

# Shared y-limits, with headroom for the percentage labels
all_densities = np.concatenate([t['density'].dropna().to_numpy() for t in ensemble_tables.values()])
y_min, y_max = all_densities.min(), all_densities.max()
margin = 0.05 * (y_max - y_min)
label_room = 0.07 * (y_max - y_min)
axes[0].set_ylim(min(y_min, DENSITY_THRESHOLD) - margin, max(y_max, DENSITY_THRESHOLD) + margin + label_room)

legend_handles = [Patch(facecolor=c, edgecolor='black', linewidth=0.8, label=label) for _, label, c in SCENARIOS]
legend_handles.append(Line2D([], [], color='black', linestyle='--', linewidth=1.2,
                             label=f'{DENSITY_THRESHOLD} kg m$^{{-3}}$ threshold'))
fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False,
           bbox_to_anchor=((left + right) / 2, top_margin + 0.035))

# Align the left edge of the panel titles with the left edge of the y-axis label
fig.canvas.draw()
for ax in axes:
    bbox = ax.yaxis.label.get_window_extent()
    x_axes = ax.transAxes.inverted().transform((bbox.x0, bbox.y0))[0]
    ax.set_title(ax.get_title(loc='left'), loc='left', x=x_axes)

save_figure(fig, 'fig08_basal20mm_density_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

# Figure D1 (Appendix D). Precipitation phase for every reanalysis-PPM pair

Thomsen River, 26 September - 10 October 2003, 12-hour accumulations. B00 is shown in its four phases (read from the raw CaSRv3.2 extraction);
the other PPMs only provide rain and snow. ERA5 with B00 does not exist.

In [ ]:
PPM_ROWS = ['J18', 'B00', 'HP13', 'W19']
UNAVAILABLE_PAIRS = {('B00', 'ERA5')}


def phase_bars(ppm, reanalysis, start, end, freq='12h'):
    """Precipitation (mm per `freq`) by phase for one reanalysis-PPM pair, at Thomsen River."""
    if ppm == 'B00':
        product = reanalysis.split('_')[1]                       # 'analysis' or 'prediction'
        hourly = read_b00_event_data(product).loc[start:end].drop(columns='Temperature')
    else:
        ds = ppm_ds.sel(location='THOMSEN_RIVER', forecast=reanalysis, PPM=ppm, time=slice(start, end))
        hourly = pd.DataFrame({'Rain': ds['PRERN'].values * 3600, 'Snow': ds['PRESNO'].values * 3600},
                              index=pd.to_datetime(ds.time.values))
    return hourly.groupby(pd.Grouper(freq=freq)).sum()


def reanalysis_temperature(reanalysis, start, end):
    ds = ppm_ds.sel(location='THOMSEN_RIVER', forecast=reanalysis, PPM='HP13', time=slice(start, end))
    return pd.Series(ds['TA'].values, index=pd.to_datetime(ds.time.values))      # identical for all PPMs

In [ ]:
use_manuscript_style()
start, end = ROS_WINDOW_2003
model_temperature = {r: reanalysis_temperature(r, start, end) for r in REANALYSES}
all_temperatures = np.concatenate([thomsen_lws['temp'].to_numpy()] + [t.to_numpy() for t in model_temperature.values()])
temperature_limits = (np.nanmin(all_temperatures) - 1, np.nanmax(all_temperatures) + 1)   # same range on every twin axis

with plt.style.context('ggplot'):
    fig, axes = plt.subplots(len(PPM_ROWS), len(REANALYSES), figsize=(4 * len(REANALYSES), 2.5 * len(PPM_ROWS)),
                             sharex=True, sharey=True)

    for row, ppm in enumerate(PPM_ROWS):
        for col, reanalysis in enumerate(REANALYSES):
            ax = axes[row, col]
            ax_twin = ax.twinx()
            ax.set_ylim([0, 4])
            ax_twin.set_ylim(temperature_limits)
            ax_twin.grid(False)
            ax_twin.patch.set_visible(False)

            if (ppm, reanalysis) in UNAVAILABLE_PAIRS:
                ax.set_xticks([])
                ax.grid(False)
                ax.text(0.5, 0.5, 'Unavailable', ha='center', va='center', transform=ax.transAxes, fontsize=11, color='grey')
            else:
                bars = phase_bars(ppm, reanalysis, start, end).loc[start:end]
                bottom = np.zeros(len(bars))
                for phase in bars.columns:
                    ax.bar(mdates.date2num(bars.index), bars[phase], bottom=bottom, width=0.45,
                           color=PHASE_COLORS[phase], align='edge', zorder=3)
                    bottom += bars[phase].to_numpy()
                shade_ros_detections(ax, ROS_DETECTIONS_2003, color='red', alpha=0.12)
                ax.grid(True, axis='y', zorder=1, alpha=0.4)

                ax_twin.plot(thomsen_lws['time'], thomsen_lws['temp'], color=TEMPERATURE_COLOR, linestyle='--', zorder=4, linewidth=0.8)
                ax_twin.plot(model_temperature[reanalysis].index, model_temperature[reanalysis].values,
                             color=TEMPERATURE_COLOR, zorder=4, linewidth=0.8)
                ax_twin.axhline(0, zorder=1, color='k', linestyle=':', alpha=0.3)

            if row == 0:
                ax.set_title(REANALYSIS_LABELS[reanalysis])
            if col == 0:
                ax.set_ylabel('Precipitation (mm)')
                ax.text(-0.15, 0.5, ppm, transform=ax.transAxes, rotation=90, ha='center', va='center',
                        fontsize=14, fontweight='bold')
            else:
                ax.tick_params(labelleft=False)
            if col == len(REANALYSES) - 1:
                ax_twin.set_ylabel('Temperature (°C)')
            else:
                ax_twin.tick_params(labelright=False)

    for ax in axes[-1, :]:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        for tick_label in ax.get_xticklabels(which='major'):
            tick_label.set(rotation=45, horizontalalignment='right')

temperature_handles = [Line2D([0], [0], color=TEMPERATURE_COLOR, linestyle='--', label='LWS Temperature'),
                       Line2D([0], [0], color=TEMPERATURE_COLOR, label='Meteorological Reanalysis Temperature')]
phase_handles = [mpatches.Patch(facecolor=color, label=phase) for phase, color in PHASE_COLORS.items()]
phase_handles.append(mpatches.Patch(color='red', alpha=0.2, label='ROS detections'))
fig.legend(handles=temperature_handles, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.047), frameon=False)
fig.legend(handles=phase_handles, loc='upper center', ncol=len(phase_handles), bbox_to_anchor=(0.5, 1.023), frameon=False)

fig.tight_layout()
save_figure(fig, 'figD1_precip_phase_all_pairs.png', bbox_inches='tight')
plt.show()